In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q pyzipper gdown

import pyzipper
import gdown
import os
import shutil
from unsloth import FastLanguageModel

# 1. Configuration
SHARED_LINK = "PASTE_YOUR_LINK_HERE" # Paste your Google Drive shared link here
local_zip_path = "/content/temp_adapter_encrypted.zip"
extract_dir = "/content/final_adapter"

# Key Configuration
password = "" # Your AES key used for DECRYPTION
ENCRYPTION_KEY = "" # Leave blank to default to using the same password for encryption

target_enc_key = ENCRYPTION_KEY if ENCRYPTION_KEY else password

# Helper to pull the exact file ID from a standard share link
def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

# 2. Download from Shared Link
print("Downloading encrypted master backup from Google Drive...")
file_id = get_gdrive_id(SHARED_LINK)
download_url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(download_url, local_zip_path, quiet=False)

# 3. Decrypt and unzip
print("Decrypting master backup...")
os.makedirs(extract_dir, exist_ok=True)
with pyzipper.AESZipFile(local_zip_path, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zf:
    zf.pwd = password.encode('utf-8')
    zf.extractall(extract_dir)

# --- FIX 1: Hunt down the exact nested folder containing the adapter config ---
adapter_path = extract_dir
for root, dirs, files in os.walk(extract_dir):
    if 'adapter_config.json' in files:
        adapter_path = root
        break

# 4. Load the adapter back into Unsloth
print(f"Loading model from {adapter_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

# 5. Export the GGUF locally first for fast I/O
print("Compiling GGUF locally...")
local_export_dir = "/content/temp_gguf_export"
os.makedirs(local_export_dir, exist_ok=True)
gguf_prefix = os.path.join(local_export_dir, "cpu_model")
model.save_pretrained_gguf(gguf_prefix, tokenizer, quantization_method="q4_k_m")

# --- FIX 2: Encrypt using mode: STORE and ZIP ---
print("Zipping and AES-256 encrypting the GGUF file (Mode: Store)...")
save_dir = "/content/drive/MyDrive/soulclone"
os.makedirs(save_dir, exist_ok=True)

final_zip_name = "cpu_model_gguf_encrypted.zip"
local_final_zip = os.path.join("/content", final_zip_name)
drive_final_zip = os.path.join(save_dir, final_zip_name)

# pyzipper.ZIP_STORED satisfies the "mode: store" requirement (no compression)
with pyzipper.AESZipFile(local_final_zip, 'w', compression=pyzipper.ZIP_STORED, encryption=pyzipper.WZ_AES) as zf:
    zf.setpassword(target_enc_key.encode('utf-8'))
    # Dig out the .gguf file Unsloth created
    for root, _, files in os.walk(local_export_dir):
        for file in files:
            if file.endswith(".gguf"):
                file_path = os.path.join(root, file)
                zf.write(file_path, file)

# 6. Transfer to Drive
print("Transferring encrypted ZIP to Google Drive...")
shutil.copy2(local_final_zip, drive_final_zip)

print(f"Done! Check your Google Drive at {drive_final_zip} for the encrypted zip.")

In [ ]:
import time
from google.colab import runtime

print("Pipeline complete! All files should be saved to Google Drive.")
print("The runtime will automatically disconnect and delete itself in 60 seconds to save compute units...")

# Wait 1 minute to ensure Google Drive I/O operations have fully synced
time.sleep(60)

print("Terminating runtime now.")
# This command forcefully disconnects and deletes the Colab instance
runtime.unassign()